In [ ]:
!pip install gradio transformers torch tf-keras datasets tqdm evaluate

In [ ]:
import gradio as gr
from transformers import pipeline
from datasets import load_dataset

print("Loading SQuAD dataset...")
# Load SQuAD v1.1 dataset
squad_dataset = load_dataset("squad", split="train")
print(f"Loaded {len(squad_dataset)} examples from SQuAD dataset")

# Extract first 10 examples (context, question, and answer)
squad_examples = []
for i in range(10):
    example = squad_dataset[i]
    # Get the answer text (first answer if multiple exist)
    answer_text = example['answers']['text'][0] if example['answers']['text'] else ""
    squad_examples.append([
        example['context'],
        example['question'],
        answer_text  # Ground truth answer from dataset
    ])
print(f"Extracted first {len(squad_examples)} examples with answers")

print("Loading models...")

# --- Model 1: DistilBERT (Fast, SQuAD v1.1) ---
# Good for speed, trained on answerable questions only.
model_1_name = "distilbert-base-cased-distilled-squad"
qa_pipeline_1 = pipeline("question-answering", model=model_1_name)

# --- Model 2: RoBERTa (Accurate, SQuAD v2.0) ---
# Good for accuracy, can handle "unanswerable" questions (returns empty string).
model_2_name = "deepset/roberta-base-squad2"
qa_pipeline_2 = pipeline("question-answering", model=model_2_name)

In [ ]:
# --- Model Evaluation on SQuAD Dataset ---
# This section evaluates both models on the SQuAD dataset

from evaluate import evaluator
from datasets import load_dataset

print("\n" + "="*60)
print("EVALUATING MODELS ON SQuAD DATASET")
print("="*60)

# Load entire validation set for evaluation
eval_data = load_dataset("squad", split="validation")
print(f"Loaded {len(eval_data)} examples from SQuAD validation set")

# Initialize the question-answering evaluator
task_evaluator = evaluator("question-answering")

# Evaluate Model 1 (DistilBERT - SQuAD v1.1)
print(f"\nEvaluating {model_1_name}...")
results_1 = task_evaluator.compute(
    model_or_pipeline=qa_pipeline_1,
    data=eval_data,
    metric="squad",
)

# Evaluate Model 2 (RoBERTa - SQuAD v2.0)
# Note: Model 2 is trained on SQuAD v2, but we're evaluating on SQuAD v1.1
# For SQuAD v2 evaluation, use squad_v2_format=True
print(f"\nEvaluating {model_2_name}...")
results_2 = task_evaluator.compute(
    model_or_pipeline=qa_pipeline_2,
    data=eval_data,
    metric="squad",
)

# Display results
print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"\nModel 1: {model_1_name}")
print(f"  F1 Score: {results_1['f1']:.4f}")
print(f"  Exact Match: {results_1['exact_match']:.4f}")

print(f"\nModel 2: {model_2_name}")
print(f"  F1 Score: {results_2['f1']:.4f}")
print(f"  Exact Match: {results_2['exact_match']:.4f}")

print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"F1 Score Difference: {abs(results_1['f1'] - results_2['f1']):.4f}")
print(f"Exact Match Difference: {abs(results_1['exact_match'] - results_2['exact_match']):.4f}")

if results_1['f1'] > results_2['f1']:
    print(f"\n🏆 Model 1 ({model_1_name}) has higher F1 score")
elif results_2['f1'] > results_1['f1']:
    print(f"\n🏆 Model 2 ({model_2_name}) has higher F1 score")
else:
    print(f"\n🤝 Models have similar F1 scores")

if results_1['exact_match'] > results_2['exact_match']:
    print(f"🏆 Model 1 ({model_1_name}) has higher Exact Match score")
elif results_2['exact_match'] > results_1['exact_match']:
    print(f"🏆 Model 2 ({model_2_name}) has higher Exact Match score")
else:
    print(f"🤝 Models have similar Exact Match scores")
print("="*60 + "\n")


In [ ]:
# Gradio App

def compare_models(context, question):
    # Error handling for empty inputs
    if not context or not question:
        return "N/A", 0.0, "N/A", 0.0

    # Run Model 1
    res1 = qa_pipeline_1(question=question, context=context)
    
    # Run Model 2
    res2 = qa_pipeline_2(question=question, context=context)

    # Return: Ans1, Conf1, Ans2, Conf2
    return res1['answer'], res1['score'], res2['answer'], res2['score']

# --- Gradio UI Layout ---
with gr.Blocks(title="QA Model Arena") as demo:
    gr.Markdown("# ⚔️ QA Model Arena: DistilBERT vs. RoBERTa")
    gr.Markdown("Compare a lightweight model (DistilBERT) against a robust model (RoBERTa) on the **SQuAD** dataset.")
    
    with gr.Row():
        with gr.Column(scale=1):
            context_input = gr.Textbox(lines=8, label="Context Paragraph", placeholder="Paste context here...")
            question_input = gr.Textbox(lines=2, label="Question")
            ground_truth_answer = gr.Textbox(lines=2, label="Ground Truth Answer (from SQuAD)", interactive=False)
            submit_btn = gr.Button("Compare Models", variant="primary")
        
        with gr.Column(scale=1):
            gr.Markdown(f"### 🚀 Model A: {model_1_name}")
            out_ans_1 = gr.Textbox(label="Answer")
            out_conf_1 = gr.Number(label="Confidence")
            
            gr.Markdown("---")
            
            gr.Markdown(f"### 🧠 Model B: {model_2_name}")
            out_ans_2 = gr.Textbox(label="Answer")
            out_conf_2 = gr.Number(label="Confidence")

    # Link inputs/outputs
    submit_btn.click(
        fn=compare_models,
        inputs=[context_input, question_input],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2]
    )

    # Add Examples
    gr.Examples(
        examples=squad_examples,
        inputs=[context_input, question_input, ground_truth_answer],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2],
        fn=compare_models,
        cache_examples=True,
    )

if __name__ == "__main__":
    demo.launch()